In [ ]:
!pip install groq faster-whisper gtts

In [ ]:
from groq import Groq

client = Groq(
    api_key="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"
)

SYSTEM_PROMPT = """
You are a multilingual Duolingo-style AI tutor.

Rules:
- Detect language automatically
- Reply in the same language
- Correct grammar politely
- Keep answers short
- Teach naturally
- Ask follow-up questions
- Encourage the learner
"""

key: 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

In [ ]:
def ai_tutor(user_text):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            }
        ],

        temperature=0.7,
        max_tokens=300
    )

    return response.choices[0].message.content

In [ ]:
!pip install gtts langdetect

In [ ]:
from gtts import gTTS
from langdetect import detect

def text_to_speech(text):

    try:
        language = detect(text)
    except:
        language = "en"

    output_file = "reply.mp3"

    tts = gTTS(
        text=text,
        lang=language,
        slow=False
    )

    tts.save(output_file)

    return output_file

In [ ]:
from IPython.display import Audio

def voice_to_voice(audio_path):

    # Voice -> Text
    user_text = speech_to_text(audio_path)

    print("\n USER:")
    print(user_text)

    # Text -> AI
    ai_reply = ai_tutor(user_text)

    print("\n AI:")
    print(ai_reply)

    # AI Text -> Voice
    audio_file = text_to_speech(ai_reply)

    return Audio(audio_file, autoplay=True)

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    "medium",
    device="cuda",
    compute_type="float16"
)

In [ ]:
def speech_to_text(audio_path):

    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5
    )

    text = ""

    for segment in segments:
        text += segment.text + " "

    print("Language:", info.language)

    return text.strip()

In [ ]:
from google.colab import output
from base64 import b64decode

In [ ]:
RECORD_JS = """
async function recordAudio() {

    const stream = await navigator.mediaDevices.getUserMedia({
        audio: true
    });

    const recorder = new MediaRecorder(stream);

    let chunks = [];

    recorder.ondataavailable = e => {
        chunks.push(e.data);
    };

    recorder.start();

    alert("Recording started. Speak now for 120 seconds.");

    await new Promise(resolve => setTimeout(resolve, 5000));

    recorder.stop();

    await new Promise(resolve => {
        recorder.onstop = resolve;
    });

    stream.getTracks().forEach(track => track.stop());

    const blob = new Blob(chunks, {
        type: "audio/webm"
    });

    const reader = new FileReader();

    reader.readAsDataURL(blob);

    await new Promise(resolve => {
        reader.onloadend = resolve;
    });

    return reader.result;
}

recordAudio();
"""

In [ ]:
output.eval_js("""
navigator.mediaDevices.getUserMedia({audio:true})
.then(() => "MIC OK")
.catch(err => err.toString())
""")

'MIC OK'

In [ ]:
audio_data = output.eval_js(RECORD_JS)

In [ ]:
audio_bytes = b64decode(
    audio_data.split(",")[1]
)

with open("input.webm", "wb") as f:
    f.write(audio_bytes)

print("Saved as input.webm")

Saved as input.webm


In [ ]:
import os

print(os.path.exists("input.webm"))

True


In [ ]:
user_text = speech_to_text("input.webm")

print(user_text)

Language: ko
오하요  안녕하세요  오빠가


In [ ]:
voice_to_voice("input.webm")

Language: ko

 USER:
잘자요  안녕하세요  안녕

 AI:
안녕하세요! 잘자요, 안녕은 서로 다른 인사말이에요. 잘자요는 밤에 자러 갈 때 쓰고, 안녕하세요는 처음 만나거나 인사를 할 때 쓰며, 안녕은 친구나 친한 사람에게 쓰는 인사말이에요. 잘자요는 잘 잤어요, 다음에 또 만나요!


In [ ]:
def speech_to_text(audio_path):

    segments, info = whisper_model.transcribe(
        audio_path,
        beam_size=5
    )

    text = " ".join(
        segment.text
        for segment in segments
    )

    print("Language:", info.language)

    return text